In [1]:
import sys
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, accuracy_score)

ROOT = Path(r"C:\ml_eng_prep_2026\project-1-secom-yield")
INTERIM = ROOT/"data"/"interim"
sys.path.insert(0, str(ROOT))

from src.load import load_secom
from src.features import build_feature_spec, apply_feature_spec

X_all, y, ts = load_secom()
splits = pd.read_csv(INTERIM/"split_idx.csv")
train_idx = splits.loc[splits["split"]=="train", "idx"].values

spec = build_feature_spec(X_all.loc[train_idx])
X = apply_feature_spec(X_all, spec).loc[train_idx]
y_tr = y[train_idx]
print(X.shape, y_tr.sum())

(1253, 442) 83


In [2]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows, oof = [], np.zeros(len(X))

for fold, (tr, va) in enumerate(skf.split(X, y_tr)):
    pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())])
    Xtr = pipe.fit_transform(X.iloc[tr])
    Xva = pipe.transform(X.iloc[va])

    clf = LogisticRegression(class_weight="balanced", max_iter=2000)
    clf.fit(Xtr, y_tr.iloc[tr])

    p = clf.predict_proba(Xva)[:, 1]
    pred = (p >= 0.5).astype(int)
    oof[va] = p
    yva = y_tr.iloc[va]

    rows.append({
        "fold": fold,
        "precision": precision_score(yva, pred, zero_division=0),
        "recall": recall_score(yva, pred),
        "f1": f1_score(yva, pred),
        "roc_auc": roc_auc_score(yva, p),
        "accuracy": accuracy_score(yva, pred),
    })

res = pd.DataFrame(rows)
print(res.round(3))
print(res.drop(columns="fold").agg(["mean", "std"]).round(3))

   fold  precision  recall     f1  roc_auc  accuracy
0     0      0.206   0.412  0.275    0.705     0.853
1     1      0.195   0.471  0.276    0.746     0.833
2     2      0.222   0.353  0.273    0.691     0.873
3     3      0.091   0.125  0.105    0.539     0.864
4     4      0.097   0.188  0.128    0.616     0.836
      precision  recall     f1  roc_auc  accuracy
mean      0.162   0.310  0.211    0.659     0.852
std       0.063   0.148  0.087    0.082     0.017


In [3]:
def cv_eval(model_fn, X, y, name, skf):
    rows, oof = [], np.zeros(len(X))
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler())])
        Xtr = pipe.fit_transform(X.iloc[tr])
        Xva = pipe.transform(X.iloc[va])
        ytr, yva = y.iloc[tr], y.iloc[va]

        clf = model_fn(ytr)
        clf.fit(Xtr, ytr)

        p = clf.predict_proba(Xva)[:, 1]
        pred = (p >= 0.5).astype(int)
        oof[va] = p
        rows.append({
            "model": name, "fold": fold,
            "precision": precision_score(yva, pred, zero_division=0),
            "recall": recall_score(yva, pred),
            "f1": f1_score(yva, pred),
            "roc_auc": roc_auc_score(yva, p),
            "accuracy": accuracy_score(yva, pred),
        })
    return pd.DataFrame(rows), oof

In [4]:
from xgboost import XGBClassifier
logreg_fn = lambda ytr: LogisticRegression(class_weight="balanced", max_iter=2000)

def xgb_fn(ytr):
    spw = (ytr == 0).sum() / (ytr == 1).sum()
    return XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw, eval_metric="logloss",
        random_state=42, n_jobs=-1,
    )

res_lr, oof_lr = cv_eval(logreg_fn, X, y_tr, "logreg", skf)
res_xgb, oof_xgb = cv_eval(xgb_fn, X, y_tr, "xgboost", skf)

results = pd.concat([res_lr, res_xgb])
print(results.groupby("model").agg(["mean", "std"]).round(3).T)

model           logreg  xgboost
fold      mean   2.000    2.000
          std    1.581    1.581
precision mean   0.162    0.333
          std    0.063    0.425
recall    mean   0.310    0.036
          std    0.148    0.033
f1        mean   0.211    0.062
          std    0.087    0.058
roc_auc   mean   0.659    0.661
          std    0.082    0.075
accuracy  mean   0.852    0.931
          std    0.017    0.009


In [5]:
ts_grid = np.arange(0.05, 0.96, 0.05)
sweep = pd.DataFrame([{
    "threshold": t,
    "flagged": (oof_lr >= t).sum(),
    "precision": precision_score(y_tr, oof_lr >= t, zero_division=0),
    "recall": recall_score(y_tr, oof_lr >= t),
} for t in ts_grid])
print(sweep.round(3))

    threshold  flagged  precision  recall
0        0.05      281      0.128   0.434
1        0.10      242      0.145   0.422
2        0.15      222      0.149   0.398
3        0.20      212      0.156   0.398
4        0.25      196      0.168   0.398
5        0.30      189      0.175   0.398
6        0.35      176      0.182   0.386
7        0.40      171      0.181   0.373
8        0.45      162      0.173   0.337
9        0.50      155      0.168   0.313
10       0.55      149      0.174   0.313
11       0.60      145      0.179   0.313
12       0.65      138      0.188   0.313
13       0.70      128      0.203   0.313
14       0.75      124      0.202   0.301
15       0.80      112      0.214   0.289
16       0.85      102      0.196   0.241
17       0.90       90      0.200   0.217
18       0.95       73      0.205   0.181


Threshold 0.30. Below 0.30 is dominated: recall stays flat at 0.398 while flagged rises from 189 to 281, so those extra inspections catch nothing. Above it, precision gains little (0.175 → 0.214) while recall falls to 0.289 — trading 9 real fails for 77 saved inspections, which is the wrong trade when escape cost exceeds inspection cost. At 0.30: 189 of 1,253 lots flagged, 33 of 83 fails caught. Recall ceilings at 0.43 — a model limitation, not a threshold one. Out-of-fold on training data; test set untouched.